# Step 14 -- Selected Offline RL Comparison

Runs the selected offline RL branch on Google Colab using the same selected state/action setup as the current Step 13 CARE-Sim control track.

This notebook does two things:
- trains an offline DDQN directly on the logged selected ICU data
- evaluates it on held-out logged data and compares it against the Step 13 CARE-Sim DDQN and MarkovSim DDQN when those checkpoints are available

## Before running

**Step 1:** Rebuild the upload zip locally:
```bash
python scripts/prepare_colab_upload.py
```

**Step 2:** Upload to Google Drive:
- `caresim_colab.zip` -> `MyDrive/CareAI/`
- `rl_dataset_selected.parquet` -> `MyDrive/CareAI/data/`
- `models/icu_readmit/terminal_readmit_selected/` -> `MyDrive/CareAI/models/icu_readmit/terminal_readmit_selected/`
- optional but recommended for direct comparison: `models/icu_readmit/caresim_control_selected_causal/` -> `MyDrive/CareAI/models/icu_readmit/caresim_control_selected_causal/`
- optional but recommended for direct comparison: `models/icu_readmit/markovsim_control_selected_causal/` -> `MyDrive/CareAI/models/icu_readmit/markovsim_control_selected_causal/`

**Step 3:** Runtime -> Change runtime type -> **T4 GPU**.


In [ ]:
# Cell 1: Mount Drive + check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. This run will be slower on CPU.')


In [ ]:
# Cell 2: Unzip source files + verify inputs
import os, sys, zipfile, shutil

DRIVE_ROOT = '/content/drive/MyDrive/CareAI'
ZIP_PATH = os.path.join(DRIVE_ROOT, 'caresim_colab.zip')
UNZIP_DIR = '/content/caresim_src'

DATA_PATH = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_selected.parquet')
TERMINAL_MODEL_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'terminal_readmit_selected')
CARESIM_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'caresim_control_selected_causal')
CARESIM_DDQN = os.path.join(CARESIM_DIR, 'ddqn_model.pt')
MARKOVSIM_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'markovsim_control_selected_causal')
MARKOVSIM_DDQN = os.path.join(MARKOVSIM_DIR, 'ddqn_model.pt')
OFFLINE_MODEL_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'offline_selected')
OFFLINE_REPORT_DIR = os.path.join(DRIVE_ROOT, 'reports', 'icu_readmit', 'offline_selected')

if os.path.exists(UNZIP_DIR):
    shutil.rmtree(UNZIP_DIR)
os.makedirs(UNZIP_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(UNZIP_DIR)

sys.path.insert(0, os.path.join(UNZIP_DIR, 'src'))
STEP14_SCRIPT = os.path.join(UNZIP_DIR, 'scripts', 'icu_readmit', 'step_14_offline_selected.py')

for path, name in [
    (ZIP_PATH, 'caresim_colab.zip'),
    (DATA_PATH, 'rl_dataset_selected.parquet'),
    (TERMINAL_MODEL_DIR, 'terminal_readmit_selected'),
    (STEP14_SCRIPT, 'step_14_offline_selected.py'),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing required input: {name} -> {path}')

print('Verified required inputs.')
print('CARE-Sim DDQN present: ', os.path.exists(CARESIM_DDQN))
print('MarkovSim DDQN present:', os.path.exists(MARKOVSIM_DDQN))


In [ ]:
# Cell 3: Full-run config
TRAIN_ARGS = [
    '--data', DATA_PATH,
    '--model-dir', OFFLINE_MODEL_DIR,
    '--report-dir', OFFLINE_REPORT_DIR,
    '--terminal-model-dir', TERMINAL_MODEL_DIR,
    '--caresim-ddqn-path', CARESIM_DDQN,
    '--markovsim-ddqn-path', MARKOVSIM_DDQN,
    '--severity-mode', 'handcrafted',
    '--device', device,
    '--dqn-steps', '100000',
]

EVAL_ARGS = [
    '--data', DATA_PATH,
    '--model-dir', OFFLINE_MODEL_DIR,
    '--report-dir', OFFLINE_REPORT_DIR,
    '--terminal-model-dir', TERMINAL_MODEL_DIR,
    '--caresim-ddqn-path', CARESIM_DDQN,
    '--markovsim-ddqn-path', MARKOVSIM_DDQN,
    '--severity-mode', 'handcrafted',
    '--device', device,
    '--physician-steps', '35000',
    '--reward-steps', '30000',
    '--env-steps', '60000',
]

print('Train args:', TRAIN_ARGS)
print('Eval args: ', EVAL_ARGS)


In [ ]:
# Cell 4: Train offline DDQN
import subprocess, sys

cmd = [sys.executable, STEP14_SCRIPT, 'train-ddqn', *TRAIN_ARGS]
print('Running:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Cell 5: Run held-out evaluation / comparison
import subprocess, sys

cmd = [sys.executable, STEP14_SCRIPT, 'eval', *EVAL_ARGS]
print('Running:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Cell 6: Verify outputs
import os, json

print('Offline model files saved to Drive:')
for root, dirs, files in os.walk(OFFLINE_MODEL_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        rel = os.path.relpath(path, OFFLINE_MODEL_DIR)
        print(f'  {rel:60s} {os.path.getsize(path)/1024:8.1f} KB')

print('\nOffline report files saved to Drive:')
for root, dirs, files in os.walk(OFFLINE_REPORT_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        rel = os.path.relpath(path, OFFLINE_REPORT_DIR)
        print(f'  {rel:60s} {os.path.getsize(path)/1024:8.1f} KB')

summary_path = os.path.join(OFFLINE_REPORT_DIR, 'step_14_eval_results.json')
if os.path.exists(summary_path):
    print('\nSaved summary:')
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print(json.dumps(summary.get('ope', {}), indent=2)[:4000])

print('\nDownload from Drive to these local repo folders:')
print('  CareAI/models/icu_readmit/offline_selected/')
print('  CareAI/reports/icu_readmit/offline_selected/')
